# Bakta CTS Demo

End-to-end demo of the cdm_bakta CTS tool.

- **Image:** `ghcr.io/kbaseincubator/cdm_bakta:0.1.3@sha256:ab3d636d9381fdd57a8b21e3a2403663987f57f123027fa7a773474e66d9a827`
- **Refdata UUID:** `30f8ba11-a456-408c-a9f9-7d232ba3ed8e`
- **Refdata file:** `cts-refdata/bakta/v6.0_amr20260324/bakta_db.tar.gz` (~30GB compressed, ~84GB unpacked)
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/bakta/test/v3_diamond_v220`

Bakta does full bacterial genome annotation. Input is **nucleotide FASTA** (genome assemblies, `.fna` or `.fna.gz`), unlike kofamscan which needs protein sequences.

The 0.1.3 image overlays diamond v2.2.0 over the conda-shipped diamond v2.1.21 to fix an intermittent deadlock at the pseudogene-detection alignment step. The `BAKTA_DB` env var is set in the image to `/ref_data/db`, so callers do NOT need to pass `--db` on the command line.

## 1. Setup

In [7]:
tscli = get_task_service_client()
mincli = get_minio_client()

IMAGE = "ghcr.io/kbaseincubator/cdm_bakta:0.1.3@sha256:ab3d636d9381fdd57a8b21e3a2403663987f57f123027fa7a773474e66d9a827"
OUTPUT_DIR = "cts/io/jplfaria/output/bakta/test/v3_diamond_v220"

print(tscli.whoami())

{'user': 'jplfaria', 'roles': [], 'allowed_paths': [{'path': 'cts/io/', 'perm': 'write'}]}


## 2. List input genomes

Use the same 4 test genomes the rest of the team has been using (`cts/io/gavin/test_files/`). These are nucleotide assemblies with CRC64NVME checksums.

In [8]:
input_files = []
for o in mincli.list_objects("cts", prefix="io/gavin/test_files", recursive=True):
    if o.object_name.endswith(".fna.gz") or o.object_name.endswith(".fna"):
        input_files.append(f"cts/{o.object_name}")

print(f"{len(input_files)} input genome(s):")
for f in input_files:
    print(f"  {f}")

4 input genome(s):
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000008085.1/GCA_000008085.1_ASM808v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000010565.1/GCA_000010565.1_ASM1056v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000145985.1/GCA_000145985.1_ASM14598v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000147015.1/GCA_000147015.1_ASM14701v1_genomic.fna.gz


## 3. Submit Bakta job

Bakta is heavier than kofamscan: full DB (~84GB unpacked) + lots of subprocess steps (CDS prediction, tRNA/rRNA scans, BLAST against UniRef, etc). Plan for ~20-40 min per bacterial genome at 4 CPUs.

One container per input genome (parallelizes the wall time).

In [9]:
if not input_files:
    raise RuntimeError("No .fna(.gz) inputs found. Adjust the prefix above.")

job = tscli.submit_job(
    IMAGE,
    input_files,
    OUTPUT_DIR,
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "--output", "/out",
        "--threads", "4",
        "--keep-contig-headers",
        "--skip-plot",        # skip the circular-genome PNG/SVG (slow + not needed for this test)
        "--force",            # bakta refuses to write to existing /out (CTS pre-creates it)
        tscli.insert_files(),
    ],
    num_containers=len(input_files),
    cpus=4,
    memory="32GB",
    runtime="PT4H",
)
print("Job ID:", job.id)

Job ID: 6f68bae6-c3d8-4428-a95a-1f1c75034b11


## 4. Wait for completion


In [1]:
# Block until the job finishes, printing the final status.
import json
result = job.wait_for_completion()
print(json.dumps(result, indent=2, default=str))

{
  "id": "6f68bae6-c3d8-4428-a95a-1f1c75034b11",
  "state": "complete",
  "transition_times": [
    {
      "state": "created",
      "time": "2026-05-15T07:21:18.841000Z"
    },
    {
      "state": "download_submitted",
      "time": "2026-05-15T07:21:18.915000Z"
    },
    {
      "state": "job_submitting",
      "time": "2026-05-15T07:21:31.918000Z"
    },
    {
      "state": "job_submitted",
      "time": "2026-05-15T07:21:32.243000Z"
    },
    {
      "state": "upload_submitting",
      "time": "2026-05-15T07:31:29.754000Z"
    },
    {
      "state": "upload_submitted",
      "time": "2026-05-15T07:31:30.091000Z"
    },
    {
      "state": "complete",
      "time": "2026-05-15T07:31:39.209000Z"
    }
  ],
  "user": "jplfaria",
  "admin_meta": {
    "cse_event_processing_start": "2026-05-15T07:31:39.314469+00:00",
    "cse_event_processing_no_operation": "2026-05-15T07:31:39.416295+00:00"
  }
}


## 5. Inspect output files

In [22]:
outs = job.get_job()["outputs"]
print(f"{len(outs)} total output files")
for o in outs:
    print(f"  {o['file']}")

52 total output files
  cts/io/jplfaria/output/bakta/test/v3_diamond_v220/0/GCA_000008085.1_ASM808v1_genomic.fna.log
  cts/io/jplfaria/output/bakta/test/v3_diamond_v220/0/GCA_000008085.1_ASM808v1_genomic.fna.tsv
  cts/io/jplfaria/output/bakta/test/v3_diamond_v220/0/GCA_000008085.1_ASM808v1_genomic.fna.gff3
  cts/io/jplfaria/output/bakta/test/v3_diamond_v220/0/GCA_000008085.1_ASM808v1_genomic.fna.gbff
  cts/io/jplfaria/output/bakta/test/v3_diamond_v220/0/GCA_000008085.1_ASM808v1_genomic.fna.embl
  cts/io/jplfaria/output/bakta/test/v3_diamond_v220/0/GCA_000008085.1_ASM808v1_genomic.fna.fna
  cts/io/jplfaria/output/bakta/test/v3_diamond_v220/0/GCA_000008085.1_ASM808v1_genomic.fna.ffn
  cts/io/jplfaria/output/bakta/test/v3_diamond_v220/0/GCA_000008085.1_ASM808v1_genomic.fna.faa
  cts/io/jplfaria/output/bakta/test/v3_diamond_v220/0/GCA_000008085.1_ASM808v1_genomic.fna.inference.tsv
  cts/io/jplfaria/output/bakta/test/v3_diamond_v220/0/GCA_000008085.1_ASM808v1_genomic.fna.hypotheticals.tsv
 

## 6. Read annotation TSVs as a dataframe

Bakta produces one `.tsv` per input genome with columns:
`Sequence Id, Type, Start, Stop, Strand, Locus Tag, Gene, Product, DbXrefs`.

Lines starting with `#` are headers/metadata; the actual table starts after a `#Sequence Id` header.

In [23]:
import io
import pandas as pd

tsvs = [o for o in outs if o["file"].endswith(".tsv") and "/tmp/" not in o["file"]]
print(f"{len(tsvs)} bakta .tsv annotation file(s)")

frames = []
for o in tsvs:
    bucket, key = o["file"].split("/", 1)
    obj = mincli.get_object(bucket, key)
    raw = obj.read().decode("utf-8")
    # Strip metadata comment lines, find the column header line, then parse the rest
    lines = raw.splitlines()
    header_idx = next((i for i, ln in enumerate(lines) if ln.startswith("#Sequence Id")), None)
    if header_idx is None:
        print(f"  WARN: no header line in {o['file']}")
        continue
    table = "\n".join([lines[header_idx][1:]] + lines[header_idx + 1:])
    df = pd.read_csv(io.StringIO(table), sep="\t")
    df["source_file"] = o["file"]
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"\nTotal feature rows: {len(all_df)}")
if len(all_df):
    print("\nFeature counts by Type:")
    print(all_df["Type"].value_counts())
all_df.head(20)

12 bakta .tsv annotation file(s)



Total feature rows: 16287

Feature counts by Type:
Type
cds              11698
crispr-spacer      612
crispr-repeat      612
tRNA               330
ncRNA-region       108
rRNA                28
crispr              18
ncRNA               12
tmRNA                1
Name: count, dtype: int64


   Sequence Id           Type  Start   Stop Strand    Locus Tag  Gene  \
0   AE017199.1            cds      1    879      -  AHLLAC_0001   NaN   
1   AE017199.1            cds    883   2691      +  AHLLAC_0002   NaN   
2   AE017199.1            cds   2668   3189      -  AHLLAC_0003   NaN   
3   AE017199.1            cds   3290   3748      -  AHLLAC_0004   NaN   
4   AE017199.1            cds   3745   4665      -  AHLLAC_0005   NaN   
5   AE017199.1            cds   4680   5090      +  AHLLAC_0006   NaN   
6   AE017199.1            cds   5028   5561      -  AHLLAC_0007   NaN   
7   AE017199.1            cds   5537   5707      -  AHLLAC_0008   NaN   
8   AE017199.1            cds   5759   7000      +  AHLLAC_0009   NaN   
9   AE017199.1           tRNA   7001   7074      -  AHLLAC_0010  trnI   
10  AE017199.1            cds   7105   7401      -  AHLLAC_0011   NaN   
11  AE017199.1            cds   7398   8423      -  AHLLAC_0012   NaN   
12  AE017199.1            cds   8399   8728      - 

## 7. End-to-end check

If all of the below are True, the demo run is healthy.

In [1]:
checks = {
    "job complete": job.get_job_status()["state"] == "complete",
    "has bakta .tsv files": len(tsvs) > 0,
    "non-empty annotations": len(all_df) > 0,
    "has CDS features": (all_df["Type"] == "cds").any() if len(all_df) else False,
    "has Product assignments": all_df["Product"].notna().any() if len(all_df) else False,
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")

if all(checks.values()):
    print("\nAll green.")
else:
    print("\nSomething's off, inspect outputs above.")

  [x] job complete
  [x] has bakta .tsv files
  [x] non-empty annotations
  [x] has CDS features
  [x] has Product assignments

All green.


## 8. Diagnostic (run if job errored)

In [24]:
# Error diagnostic - only run if state is error
ed = job.get_job()
print("logpath:", ed.get("logpath"))
print("error:", ed.get("error"))
print("exit codes:", job.get_exit_codes())
print("---stderr container 0---")
try:
    job.print_logs(container_num=0, stderr=True)
except Exception as e:
    print("stderr unavailable:", e)
print("---stdout container 0---")
try:
    job.print_logs(container_num=0, stderr=False)
except Exception as e:
    print("stdout unavailable:", e)


CTS returned error structure:
{'error': {'httpcode': 404, 'httpstatus': 'Not Found', 'time': '2026-05-12T18:48:49.926440+00:00', 'request_id': '3973dfd9-7634-4b27-8726-83097c679645', 'appcode': 40070, 'apperror': 'No logs available', 'message': 'Job ID b32fc4d4-e8ff-4a76-9e55-49c2f058487d has no logs available'}}


logpath: None
error: An unexpected error occurred.
exit codes: {'exit_codes': [None, None, None, None]}
---stderr container 0---
stderr unavailable: Job ID b32fc4d4-e8ff-4a76-9e55-49c2f058487d has no logs available
---stdout container 0---


CTS returned error structure:
{'error': {'httpcode': 404, 'httpstatus': 'Not Found', 'time': '2026-05-12T18:48:50.013463+00:00', 'request_id': '8fcd6995-0d22-414f-aca5-fd7153e0df74', 'appcode': 40070, 'apperror': 'No logs available', 'message': 'Job ID b32fc4d4-e8ff-4a76-9e55-49c2f058487d has no logs available'}}


stdout unavailable: Job ID b32fc4d4-e8ff-4a76-9e55-49c2f058487d has no logs available
